# 01 - Data acquisition, cleaning and EDA
**Task 1.** Downloads the UCI Heart Disease (Cleveland) file, cleans it, and explores it.
All logic lives in `cardiorisk/data.py` and `cardiorisk/eda.py`; this notebook calls the same
functions CI runs, so the figures here are identical to the ones CI produces.

In [ ]:
# Make the repo importable whether Jupyter was started in the repo root or in notebooks/
import os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "cardiorisk").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("Repo root:", ROOT)

In [ ]:
%matplotlib inline
import pandas as pd
from cardiorisk import config as C
from cardiorisk.data import download_raw, load_raw, clean_data, summarize
from cardiorisk import eda
pd.set_option("display.max_columns", 30)

## 1. Acquire
`download_raw()` skips the download if `data/raw/processed.cleveland.data` already exists.
The raw file has no header and uses `?` for missing values.

In [ ]:
raw = load_raw(download_raw())
print(raw.shape)
raw.head()

## 2. Clean
Cleaning coerces types, binarises the target (`num` 0 -> 0, 1-4 -> 1), turns impossible values
and unknown codes into missing, and drops duplicates. **It does not impute**: imputation is part
of the model pipeline so it is learned from training rows only.

In [ ]:
df = clean_data(raw)
summary = summarize(raw, df)
for k, v in summary.items():
    print(f"{k:15s} {v}")
df.to_csv(C.CLEAN_FILE, index=False)
print("\nSaved", C.CLEAN_FILE)

In [ ]:
df.describe().T.round(2)

## 3. Class balance and missing values

In [ ]:
eda.plot_class_balance(df);

In [ ]:
eda.plot_missing_values(df);

## 4. Numeric features by diagnosis

In [ ]:
eda.plot_numeric_distributions(df);

In [ ]:
eda.plot_outlier_boxplots(df);

## 5. Disease rate per category
Bars above the dashed line mark categories with more disease than average.

In [ ]:
eda.plot_disease_rate_by_category(df);

## 6. Correlations

In [ ]:
eda.plot_correlation_heatmap(df);

## 7. Computed findings (use these numbers in the report)

In [ ]:
findings = eda.key_findings(df)
for k, v in findings.items():
    print(f"{k:28s} {v}")

In [ ]:
# Save every EDA figure to reports/figures/ for the report
paths = eda.generate_all(df)
print(*[p.name for p in paths], sep="\n")